<a href="https://colab.research.google.com/github/chappypro123/Unicafam_Modelos_Analiticos_Indra/blob/main/Ejercicio1_Javier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

EJERCICIOS DE BGI DATA

Ejercicio_1

In [13]:
import pandas as pd
import numpy as np

np.random.seed(321)
n = 10

calificacion = pd.DataFrame({
    "edad": np.random.choice([21, 32, 32, 23, 23], n),
    "estrato": np.random.choice([1, 2, 3, 4, 5], n),
    "programa": np.random.choice(["sociales", "matematicas", "religión", "algebra", "ciencia"], n),
    "promedio": np.random.choice([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], n),
    "materias_aprobadas": np.random.choice([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], n),
    "jornada": np.random.choice(["mañana", "mediodía", "tarde", "noche", "amanecer"], n)
})

print(calificacion.shape)
print(calificacion.head(5))

(10, 6)
   edad  estrato     programa  promedio  materias_aprobadas   jornada
0    23        1     religión         2                   8     noche
1    32        4      algebra         1                   1    mañana
2    23        4  matematicas         5                   9     tarde
3    32        2     religión         5                   8    mañana
4    21        4      ciencia         4                   2  amanecer


In [15]:
#Tipo de dato de cada columna
print(calificacion.dtypes)

edad                   int64
estrato                int64
programa              object
promedio               int64
materias_aprobadas     int64
jornada               object
dtype: object


In [17]:
print(calificacion.nunique())


edad                  3
estrato               5
programa              4
promedio              6
materias_aprobadas    7
jornada               5
dtype: int64


In [18]:
for col in calificacion.columns:
    serie = calificacion[col]
    unicos = serie.nunique()
    if pd.api.types.is_datetime64_any_dtype(serie):
        clase = "Fecha"
    elif pd.api.types.is_numeric_dtype(serie):
        clase = "Cuantitativa"
    else:
        clase = "Cualitativa"
    print(f"{col:10} {clase:13} valores únicos: {unicos}")

edad       Cuantitativa  valores únicos: 3
estrato    Cuantitativa  valores únicos: 5
programa   Cualitativa   valores únicos: 4
promedio   Cuantitativa  valores únicos: 6
materias_aprobadas Cuantitativa  valores únicos: 7
jornada    Cualitativa   valores únicos: 5


La variable estrato es Cualitativa debido a que categoriza a la sociedad por niveles



Fase 1: Objetivo y KPI de éxito

• Objetivo de negocio: Disminuir la proporción de pedidos entregados fuera del tiempo prometido.
• KPI de éxito: Tasa de entregas a tiempo, medida como el porcentaje de pedidos entregados dentro del tiempo estimado.

Fase 2: 5 datos necesarios

1. Ciclo del pedido: Hora exacta de creación, aceptación del restaurante, recogida por el domiciliario y entrega final.
2. Coordenadas GPS (origen y destino): Ubicación del comercio y del cliente para calcular distancias reales y zonas de Bogotá.
3. Tiempo de preparación del restaurante:
4. Disponibilidad de repartidores
5. Condiciones externas (clima y tráfico).

Fase 3: 2 problemas de calidad de datos esperados

1. Registros incompletos o fallos de GPS.
2. Tiempos inconsistentes por error humano: Repartidores o restaurantes que marcan "pedido listo" o "entregado" antes o después de que realmente ocurra, creando duraciones negativas, iguales a cero o valores atípicos.

Fases 4–6: Qué predecir y cómo lo usaría la empresa

• Qué se predice: La probabilidad de retraso de un pedido o los minutos exactos de demora justo en el momento en que el usuario confirma la compra.
• Cómo lo usa la empresa:
	• Ajuste dinámico en la app: Mostrarle al cliente un tiempo estimado de entrega más realista antes de pagar para evitar frustración y cancelaciones.
	• Priorización operativa: Asignar automáticamente el pedido al domiciliario más cercano o enviar alertas tempranas al equipo de soporte para gestionar los pedidos con alto riesgo de demora.


In [28]:
np.random.seed(42)
n = 200
ventas = pd.DataFrame({
    "fecha": pd.date_range("2026-01-01", periods=n, freq="D"),
    "ciudad": np.random.choice(["Bogotá", "Medellín", "Cali", "Barranquilla"], n),
    "canal": np.random.choice(["Tienda", "Web", "App"], n),
    "unidades": np.random.randint(1, 20, n),
    "precio": np.random.choice([25000, 48000, 99000], n),
})
ventas["total"] = ventas["unidades"] * ventas["precio"]
print(ventas.shape)
print(ventas.head(3))


(200, 6)
       fecha        ciudad   canal  unidades  precio   total
0 2026-01-01          Cali     App        17   48000  816000
1 2026-01-02  Barranquilla     App        14   48000  672000
2 2026-01-03        Bogotá  Tienda         3   25000   75000


In [31]:
por_canal = ventas.groupby("canal")["total"].sum() / 1e6
print("Millones por canal:", por_canal.round(1).to_dict())


Millones por canal: {'App': 34.0, 'Tienda': 44.8, 'Web': 25.9}


In [29]:
ticket = ventas.groupby("ciudad")["total"].mean().round(0)
print("Ticket promedio por ciudad:")
print(ticket)



Ticket promedio por ciudad:
ciudad
Barranquilla    536926.0
Bogotá          516783.0
Cali            458981.0
Medellín        589891.0
Name: total, dtype: float64


In [30]:
ventas["dia"] = ventas["fecha"].dt.day_name()
print("Mejor día:", ventas.groupby("dia")["total"].sum().idxmax())

Mejor día: Friday


Querido gerente el día el mejor día de ventas es el viernes y las ventas en la tienda lideran las ventas